In [2]:
import pandas as pd
import pathlib
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
folder = pathlib.Path("/home/elies/data_viz/data/UN_tourism_data")

file_paths = list(folder.glob("**/*.xlsx"))

In [4]:
data_list = []

for i, path in enumerate(file_paths):
    data_i = pd.read_excel(path, sheet_name="Data")
    data_list.append(data_i)

In [5]:
tourism = pd.concat(data_list, sort=False)

In [19]:
stats = tourism["indicator_label"].unique()

df = pd.DataFrame({"indicator_label": stats})
print(df.to_string(index=False))

                                                                                                                               indicator_label
                        Implementation of standard accounting tools to monitor the economic and environmental aspects of tourism (SEEA tables)
   Implementation of standard accounting tools to monitor the economic and environmental aspects of tourism (Tourism Satellite Account tables)
                   Implementation of standard accounting tools to monitor the economic and environmental aspects of tourism (number of tables)
                                                                                           Tourism direct GDP as a proportion of total GDP (%)
                                                     employment in tourism industries - accommodation - other - male - self-employed - persons
                                                             employment in tourism industries - accommodation - other - male - total - persons

In [20]:
df = tourism.groupby("indicator_code")["reporter_area_code"].nunique().reset_index(name="reporter_area_code").sort_values("reporter_area_code")
print(df.to_string(index=False))

               indicator_code  reporter_area_code
EMPL_AIRT_TOTL_FEML_SELF_PRSN                   1
EMPL_AIRT_TOTL_MALE_SELF_PRSN                   4
EMPL_AIRT_TOTL_TOTL_SELF_PRSN                   5
EMPL_WATR_TOTL_FEML_SELF_PRSN                   5
EMPL_RAIL_TOTL_MALE_SELF_PRSN                   6
EMPL_RAIL_TOTL_TOTL_SELF_PRSN                   7
EMPL_EQUP_TOTL_FEML_SELF_PRSN                   7
EMPL_EQUP_TOTL_MALE_SELF_PRSN                  12
EMPL_EQUP_TOTL_TOTL_SELF_PRSN                  13
EMPL_AGCY_TOTL_FEML_SELF_PRSN                  14
EMPL_WATR_TOTL_TOTL_SELF_PRSN                  18
EMPL_WATR_TOTL_MALE_SELF_PRSN                  18
EMPL_SPRT_TOTL_FEML_SELF_PRSN                  18
EMPL_ROAD_TOTL_FEML_SELF_PRSN                  21
EMPL_ACCM_HOTL_FEML_SELF_PRSN                  21
EMPL_CULT_TOTL_FEML_SELF_PRSN                  22
EMPL_AGCY_TOTL_MALE_SELF_PRSN                  24
EMPL_ACCM_OTHR_FEML_SELF_PRSN                  27
EMPL_RAIL_TOTL_FEML_TOTL_PRSN                  28


In [7]:
# According to the Composition of UN Tourism region, add regions to the data
regions = pd.read_excel("data/UN_Tourism_regions_composition.xlsx", header=4)
regions = regions[["UNWTO_region_label_English", "area_label_M49_EN"]]
regions = regions.rename(columns={"UNWTO_region_label_English" : "region",
                                  "area_label_M49_EN" : "reporter_area_label"}
                        )

tourism = pd.merge(tourism, regions, on = "reporter_area_label", how = "left")

In [8]:
region_sums = tourism.groupby(["region", "indicator_code", "indicator_label", "year"]
                             )["value"].sum().reset_index()

In [85]:
employment_by_region = (
    (region_sums["indicator_code"] == "EMPL_TOTL_TOTL_TOTL_TOTL_PRSN") &
    (region_sums["year"] < 2024)
)

employment_europe = (
    (tourism["indicator_code"] == "EMPL_TOTL_TOTL_TOTL_TOTL_PRSN") &
    (tourism["region"] == "Europe")
)

visitors_by_region = (
    (region_sums["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
    (region_sums["year"] < 2024)
)

visitors_europe = (
    (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
    (tourism["region"] == "Europe")
)

expenditure_by_region = (
    (region_sums["indicator_code"] == "INBD_EXPD_BPAY_TOTL_VSTR") &
    (region_sums["year"] < 2024)
)

expenditure_europe = (
    (tourism["indicator_code"] == "INBD_EXPD_BPAY_TOTL_VSTR") &
    (tourism["region"] == "Europe")
)

In [86]:
def employment(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Employment in Tourism (Number of People)"),
        color = "region:N",
        tooltip = ["region"]
    )

    return chart

def visitors(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Number of Inbound Tourists (Number of People)"),
        color = "region:N",
        tooltip = ["region"]
    )

    return chart

def expenditure(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Tourist Expenditures (Millions USD)"),
        color = "region:N",
        tooltip = ["region"]
    )

    return chart

# Render chart
employment(region_sums[employment_by_region]) | expenditure(region_sums[expenditure_by_region])

alt.HConcatChart(...)

In [87]:
def employment_region(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Employment in Tourism (Number of People)"),
        color = "reporter_area_label:N",
        tooltip = ["reporter_area_label"]
    )

    return chart

def visitors_region(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Number of Inbound Visitors (Number of People)"),
        color = "reporter_area_label:N",
        tooltip = ["reporter_area_label"]
    )

    return chart

def expenditure_region(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Tourist Expenditures (Millions USD)"),
        color = "reporter_area_label:N",
        tooltip = ["reporter_area_label"]
    )

    return chart

# Render chart
employment_region(tourism[employment_europe]) | expenditure_region(tourism[expenditure_europe])

alt.HConcatChart(...)

In [109]:
visitors_country = (
    (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
    (tourism["reporter_area_label"] == "Peru")
)

def visitors_country_chart(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Number of Inbound Visitors (Number of People)")
    )

    return chart

# Render chart
visitors_country_chart(tourism[visitors_country])

alt.Chart(...)

In [108]:
data = tourism[
        (tourism["indicator_code"] == "INBD_EXPD_BPAY_TOTL_VSTR") &
        (tourism["year"].isin([2013, 2023])) &
        (~tourism["region"].isna())
    ]

country_growth = (
    data.pivot_table(
        index = "reporter_area_label",
        columns = "year",
        values = "value"
    ).dropna(subset=[2013, 2023]).reset_index()
)

# Calculate absolute growth
country_growth["growth"] = (country_growth[2023] - country_growth[2013]) / country_growth[2013] * 100

# Select the 5 countries with the most growth
top5 = country_growth.nlargest(5, "growth")["reporter_area_label"]

def visitors_20_year_change(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:Q").title("Year"),
        alt.Y("value").title("Number of Inbound Visitors (Number of People)"),
        color = "reporter_area_label:N",
        tooltip = ["reporter_area_label"]
    )

    return chart

visitors_20_year_change(data[data["reporter_area_label"].isin(top5)])

alt.Chart(...)

In [ ]:
# Charts below this for use in milestone 2

In [19]:
# Set a default theme for all charts
def my_theme():
    colorSchemes = {
        "category-6" : ['#ff7f0e', '#9467bd', '#2ca02c', '#d62728', '#1f77b4', '#e377c2']
    }
    
    return {
        "config" : {
            "axis": {
                "grid": False,
                "titleFontWeight" : "normal",
                "titlePadding" : 15
            },
            "view": {
                "stroke": None,
                "width" : 450
            },
            "range" : {
                "category" : colorSchemes['category-6']
            }
        }
    }

alt.themes.register('my_theme', my_theme)
alt.theme.enable('my_theme')

ThemeRegistry.enable('my_theme')

In [141]:
# Histogram, total visitors, 2022, all countries
data = tourism[
        (tourism["year"] == 2022) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

def hist_1(df):
    chart = alt.Chart(df).mark_bar(color = "#c77548").encode(
        alt.X("value:Q", bin = True).title("Inbound Tourists (Thousands of Trips)"),
        alt.Y("count()").title("Count of Countries")
    ).properties(
        title = {
            "text" : [
                    "Worldwide Tourism",
                    "2022"
            ]
        }
    )

    return chart

# Render chart
hist_1(data)

alt.Chart(...)

In [111]:
# Scatter plot, expenditure x employment, 2022, by country
data = tourism[
        (tourism["year"] == 2022) &
        (tourism["indicator_code"].isin(["INBD_EXPD_BPAY_TOTL_VSTR", "EMPL_TOTL_TOTL_TOTL_TOTL_PRSN"])) &
        (tourism["region"].notna())
    ]

data_wide = (
    data.pivot_table(
        index = ["reporter_area_label", "region"],
        columns = "indicator_code",
        values = "value"
    ).dropna().reset_index()
)

def scatter_1(df):
    chart = alt.Chart(df).mark_point(filled = True).encode(
        alt.X("INBD_EXPD_BPAY_TOTL_VSTR:Q", axis = alt.Axis(format = "$,.0f")).title("Tourist Expenditures (Millions USD)"),
        alt.Y("EMPL_TOTL_TOTL_TOTL_TOTL_PRSN:Q").title("People Employed in Tourism"),
        color = alt.Color("region:N", title = "Region"),
        tooltip = alt.Tooltip("reporter_area_label", title = "")
    ).properties(
        title = {
            "text" : [
                    "Employment and Tourist Spending by Region",
                    "2022"
            ]
        }
    ).interactive()
    # ).facet(
    #     facet = alt.Facet("region:N", title="Region"),
    #     columns = 1
    # ).resolve_scale(
    #     x="independent",
    #     y="independent"
    # )

    return chart

# Render chart
scatter_1(data_wide)

alt.Chart(...)

In [34]:
# Scatter plot, capacity x visitors, 2022, by country
data = tourism[
        (tourism["year"] == 2022) &
        (tourism["indicator_code"].isin(["CPTY_ACCM_HOTL_ROOM", "INBD_ACCM_HOTL_NGHT"])) &
        (tourism["region"].notna())
    ]

data_wide = (
    data.pivot_table(
        index = ["reporter_area_label", "region"],
        columns = "indicator_code",
        values = "value"
    ).dropna().reset_index()
)

data_wide["INBD_ACCM_HOTL_NGHT"] = data_wide["INBD_ACCM_HOTL_NGHT"] * 1000

def scatter_2(df):
    chart = alt.Chart(df).mark_point(filled = True, color = "#c77548").encode(
        alt.X("CPTY_ACCM_HOTL_ROOM:Q").title("Hotel Rooms Available"),
        alt.Y("INBD_ACCM_HOTL_NGHT:Q").title("Overnight Guests"),
        tooltip = alt.Tooltip("reporter_area_label", title = "")
    ).properties(
        title = {
            "text" : [
                    "Rooms Available and Overnight Guests",
                    "2022"
            ]
        }
    )

    line = alt.Chart(df).transform_regression(
        "CPTY_ACCM_HOTL_ROOM",
        "INBD_ACCM_HOTL_NGHT"
    ).mark_line(
        color="#e4e4e4",
        size=2,
        strokeDash=[8,5]
    ).encode(
        alt.X("CPTY_ACCM_HOTL_ROOM:Q"),
        alt.Y("INBD_ACCM_HOTL_NGHT:Q")
    )

    return (chart + line).interactive()

# Render chart
scatter_2(data_wide)

alt.LayerChart(...)

In [90]:
# Scatter plot, inbound x outbound, 2019, by country
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"].isin(["INBD_TRIP_TOTL_TOTL_VSTR", "OUTB_TRIP_TOTL_TOTL_VSTR"])) &
        (tourism["region"].notna())
    ]

data_wide = (
    data.pivot_table(
        index = ["reporter_area_label", "region"],
        columns = "indicator_code",
        values = "value"
    ).dropna().reset_index()
)

def scatter_3(df):
    chart = alt.Chart(df).mark_point(filled = True).encode(
        alt.X("INBD_TRIP_TOTL_TOTL_VSTR:Q").title("Inbound Tourists (Thousands of Trips)"),
        alt.Y("OUTB_TRIP_TOTL_TOTL_VSTR:Q").title("Outbound Tourists (Thousands of Trips)"),
        color = alt.Color("region:N", title = "Region"),
        tooltip = alt.Tooltip("reporter_area_label", title = "")
    )

    shading = alt.Chart(
            pd.DataFrame({
                "x": [0, 250000],
                "y": [0, 180000]
            })
        ).mark_area(
            color="lightgray",
            opacity=0.2
        ).encode(
            x="x:Q",
            y="y:Q"
        ).properties(
            title = {
                "text" : [
                        "In and Out Tourism by Country",
                        "2019"
                ]
            }
        )
    
    return (shading + chart).interactive()

# Render chart
scatter_3(data_wide)

alt.LayerChart(...)

In [162]:
# Line chart, total visitors, 2012-2022, by region
data = tourism[
        (tourism["year"].between(2012, 2022)) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

data = data.groupby("reporter_area_label").filter(
    lambda g: g["value"].notna().all()
    and g["year"].nunique() == 11
)

region_sums = data.groupby(["region", "indicator_code", "year"]
                             )["value"].sum().reset_index()

def line_1(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:O").title("Year"),
        alt.Y("value:Q").title("Inbound Tourists (Thousands of Trips)"),
        color = alt.Color("region:N", title = "Region")
    ).properties(title = "Impact of COVID on Worldwide Tourism")

    covid_shading = pd.DataFrame({
        "x1": [2020],
        "x2": [2021]
    })

    shading = alt.Chart(covid_shading).mark_rect(opacity=0.15, color = "grey").encode(
        x="x1:O",
        x2="x2"
    )

    return chart + shading

# Render chart
line_1(region_sums)

alt.LayerChart(...)

In [166]:
# Line chart, total visitors, 2008-2022, Ukraine
data = tourism[
        (tourism["year"].between(2008, 2022)) &
        (tourism["indicator_code"].isin(["INBD_TRIP_TOTL_TOTL_VSTR", "OUTB_TRIP_TOTL_TOTL_VSTR"])) &
        (tourism["region"].notna()) &
        (tourism["reporter_area_label"] == "Ukraine")
    ]

def line_2(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:O").title("Year"),
        alt.Y("value:Q").title("Tourists (Thousands of Trips)"),
        color = alt.Color("indicator_code:N", title = "Direction")
    ).properties(title = "Impact of War on Tourism in Ukraine")

    war_shading = pd.DataFrame({
        "x1": [2014],
        "x2": [2022]
    })

    shading = alt.Chart(war_shading).mark_rect(opacity=0.1, color = "grey").encode(
        x="x1:O",
        x2="x2"
    )

    return chart + shading

# Render chart
line_2(data)

alt.LayerChart(...)

In [175]:
# Line chart, average length of stay, all years, by region
data = tourism[
        (tourism["indicator_code"] == "CPTY_ACCM_HOTL_STAY_AVRG") &
        (tourism["region"].notna())
    ]

region_sums = data.groupby(["region", "indicator_code", "year"]
                             )["value"].mean().reset_index()

def line_3(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:O").title("Year"),
        alt.Y("value:Q").title("Average Length of Stay (Nights)"),
        color = alt.Color("region:N", title = "Region")
    ).properties(title = "Rise in Average Length of Stay in South Asia")

    return chart

# Render chart
line_3(region_sums)

alt.Chart(...)

In [191]:
# Heatmap, inbound visitors, 2019, by region x region
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"] == "INBD_TRIP_REGN_TOUR") &
        (tourism["region"].notna())
    ]

data["from_region"] = data["partner_area_label"].str.replace(" (UNWTO total)", "")

data = data[data["region"] != data["from_region"]]

region_sums = data.groupby(["region", "from_region"]
                             )["value"].sum().reset_index()

# Overnight visitors, thousand trips
def heat_1(df):
    chart = alt.Chart(df).mark_rect().encode(
        alt.X("region:N").title("To Region"),
        alt.Y("from_region:N").title("From Region"),
        color = alt.Color("value:Q", title = ["Overnight Visitors", "(Thousands of Trips)"]),
        tooltip = [alt.Tooltip("value:Q", title = "Total Trips:", format = ",.0f")]
    ).properties(height = 300,
            title = {
                "text" : [
                        "Flow of Tourists Between Regions",
                        "2019"
                ]
            }
    )

    return chart

# Render chart
heat_1(region_sums)

alt.Chart(...)

In [32]:
# Box plot, tourism GDP, 2019, by country and region
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"] == "ST_GDP_ZS") &
        (tourism["region"].notna())
    ]

data["value"] = data["value"] / 100

# data = data.sort_values(["region", "value"], ascending=[False, False])
# sort_order = data["reporter_area_label"].tolist()

def box_1(df):
    chart = alt.Chart(df).mark_boxplot().encode(
        alt.X("value:Q", axis = alt.Axis(format = "%")).title("Tourism Direct GDP as a Proportion of Total GDP (%)"),
        alt.Y("region:N").title("Region"),
        color = alt.Color("region:N", legend = None),
        tooltip = ["reporter_area_label"]
    ).properties(
            title = {
                "text" : [
                        "Tourism Proportion of GDP by Region",
                        "2019"
                ]
            }
    )

    return chart

# Render chart
box_1(data)

alt.Chart(...)

In [ ]:
# Stacked bar plot, inbound visitors by purpose, 2019, by region
# Come back to this, maybe as lollipop chart
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"].isin(["INBD_TRIP_PRPS_BNSS_TOUR", "INBD_TRIP_PRPS_PERS_TOUR"])) &
        (tourism["region"].notna())
    ]

def bar_2(df):
    chart = alt.Chart(df).mark_bar().encode(
        alt.X("reporter_area_label:N").title("Region"),
        alt.Y("sum(value):Q", stack = "normalize").title("Inbound Tourism (Thousands of Trips)"),
        color = "indicator_label:N"
    ).facet(
        facet = alt.Facet("region:N", title="Region"),
        columns = 1
    ).resolve_scale(
        x="independent"
    )

    return chart

# Render chart
bar_2(data)

In [81]:
# Stacked bar plot, inbound visitors by transport, 2019, by region
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"].isin(["INBD_TRIP_TRPT_AIRT_TOUR",
                                         "INBD_TRIP_TRPT_WTER_TOUR", 
                                         "INBD_TRIP_TRPT_LAND_TOUR",
                                         "INBD_TRIP_TOTL_TOTL_VSTR"
                                        ]
                                       )) &
        (tourism["region"].notna())
    ]

wide = (
    data.pivot_table(
        index=["reporter_area_label", "region"],
        columns="indicator_code",
        values="value"
    ).reset_index()
)

wide["percent_water"] = wide["INBD_TRIP_TRPT_WTER_TOUR"] / wide["INBD_TRIP_TOTL_TOTL_VSTR"]
wide = wide[wide["percent_water"].isin(wide["percent_water"].nlargest(20))]

def bar_3(df):
    base = alt.Chart(df).encode(
        alt.Y("reporter_area_label:N", sort = "-x").title("Country"),
        alt.X("percent_water:Q", axis=alt.Axis(format = ".0%")).title("Inbound Tourists Arriving by Water (%)"),
        tooltip = [alt.Tooltip("percent_water:Q", title = "Water Arrivals:", format = ".0%")]
    ).properties(
            title = {
                "text" : [
                        "Countries with Most Tourists Arriving by Water",
                        "2019"
                ]
            }
    )

    line = base.mark_rule(color = "blue")
    point = base.mark_point(color = "blue", opacity = 1, filled = True)
    
    return line + point

# Render chart
bar_3(wide)

alt.LayerChart(...)

In [89]:
# Stacked bar plot, employment by industry and gender, 2019, by country and region
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["region"].notna())
    ]

data[["industry", "gender"]] = data["indicator_label"].str.extract(
    r"employment in tourism industries - (.*?) - total - (female|male|total) - total - persons"
)

data = data[
    (data["industry"] != "total") &
    (data["industry"].notna()) &
    (data["gender"].isin(["male", "female"]))
]

sort_order = (
    data.pivot_table(
        index="industry",
        columns="gender",
        values="value"
    )
    .assign(pct_female=lambda x: x["female"] / (x["female"] + x["male"]))
    .sort_values("pct_female", ascending=False)
    .index.tolist()
)

def bar_4(df):
    chart = alt.Chart(df).mark_bar().encode(
        alt.Y("industry:N", sort=sort_order).title("Tourism Industry"),
        alt.X("sum(value):Q", stack = "normalize").title("Worldwide Employment"),
        color = alt.Color("gender:N", title = "Gender",
                         scale=alt.Scale(
                            domain=["female", "male"],
                            range=["pink", "#7DB1E8"]
                        ))
    ).properties(
            title = {
                "text" : [
                        "Employment by Gender in Main Tourism Industries",
                        "2019"
                ]
            }
    )

    return chart

# Render chart
bar_4(data)

alt.Chart(...)

In [35]:
# Bump chart, total visitors, 2022, all countries
data = tourism[
        (tourism["year"].between(2004, 2019)) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

top10 = data.groupby("reporter_area_label")["value"].sum().nlargest(10)

data = data[data["reporter_area_label"].isin(top10.index.tolist())]

def bump_1(df):
    chart = alt.Chart(df).mark_line(point=True).encode(
        alt.X("year:O").title("Year"),
        alt.Y("rank:O").title("Most Visited Countries"),
        color = alt.Color("reporter_area_label:N", title = "")
    ).transform_window(
        rank="rank()",
        sort=[alt.SortField("value", order="descending")],
        groupby=["year"]
    ).properties(title = "Top 10 Most 'Touristy' Countries")

    return chart

# Render chart
bump_1(data)

alt.Chart(...)